In [3]:
import optuna
import os
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
import pandas as pd

root = Path(os.getcwd()).parent / 'inputs' / 'cleaned_data.csv'

full = pd.read_csv(root)
train_df = pd.read_csv("../inputs/train.csv")
x_train = full.iloc[:len(train_df)]
y_train = np.log1p(train_df["SalePrice"])

In [4]:
def optuna_objective(trial):
    # 修正suggest_int的步长：Optuna的suggest_int步长不是直接写在第三个参数，要用step
    a = trial.suggest_int("n_estimators", 90, 130, step=10)  # 步长10，和你之前的范围对应
    b = trial.suggest_int("max_depth", 15, 25, step=1)
    c = trial.suggest_float("min_impurity_decrease", 0, 3, step=0.1)  # 建议用float，这个参数是连续值
    d = trial.suggest_int("min_samples_split", 2, 8, step=2)

    rf = RandomForestRegressor(
        n_estimators=a,
        max_depth=b,
        min_impurity_decrease=c,
        min_samples_split=d,
        n_jobs=8,
        random_state=1412
    )

    result = cross_validate(
        rf, x_train, y_train,
        scoring="neg_root_mean_squared_error",
        cv=5
    )

    return abs(result["test_score"].mean())

In [5]:
def tuna_optimize():
    study = optuna.create_study(
        sampler=optuna.samplers.TPESampler(seed=1412),  # 固定随机种子，结果可复现
        direction="minimize"
    )
    # 运行50次试验，带进度条
    study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)
    return study.best_params, study.best_value

In [6]:
best_params, best_rmse = tuna_optimize()
print("最优参数：", best_params)

[I 2026-04-16 20:42:10,363] A new study created in memory with name: no-name-c382736e-73c4-4efe-be89-bd35bc2c514d
Best trial: 0. Best value: 0.399219:   2%|▏         | 1/50 [00:00<00:36,  1.33it/s]

[I 2026-04-16 20:42:11,118] Trial 0 finished with value: 0.39921940810275525 and parameters: {'n_estimators': 130, 'max_depth': 23, 'min_impurity_decrease': 0.1, 'min_samples_split': 4}. Best is trial 0 with value: 0.39921940810275525.


Best trial: 0. Best value: 0.399219:   4%|▍         | 2/50 [00:01<00:32,  1.49it/s]

[I 2026-04-16 20:42:11,732] Trial 1 finished with value: 0.39922317878939895 and parameters: {'n_estimators': 100, 'max_depth': 23, 'min_impurity_decrease': 2.8000000000000003, 'min_samples_split': 8}. Best is trial 0 with value: 0.39921940810275525.


Best trial: 0. Best value: 0.399219:   6%|▌         | 3/50 [00:02<00:30,  1.52it/s]

...

[I 2026-04-16 20:42:50,564] Trial 47 finished with value: 0.3992184148586918 and parameters: {'n_estimators': 110, 'max_depth': 23, 'min_impurity_decrease': 0.7000000000000001, 'min_samples_split': 4}. Best is trial 5 with value: 0.39921841485869175.


Best trial: 5. Best value: 0.399218:  98%|█████████▊| 49/50 [00:40<00:00,  1.53it/s]

[I 2026-04-16 20:42:51,111] Trial 48 finished with value: 0.39922631791025764 and parameters: {'n_estimators': 90, 'max_depth': 24, 'min_impurity_decrease': 1.3, 'min_samples_split': 2}. Best is trial 5 with value: 0.39921841485869175.


Best trial: 5. Best value: 0.399218: 100%|██████████| 50/50 [00:41<00:00,  1.21it/s]

[I 2026-04-16 20:42:51,739] Trial 49 finished with value: 0.3992231787893989 and parameters: {'n_estimators': 100, 'max_depth': 16, 'min_impurity_decrease': 0.9, 'min_samples_split': 4}. Best is trial 5 with value: 0.39921841485869175.
最优参数： {'n_estimators': 110, 'max_depth': 25, 'min_impurity_decrease': 2.2, 'min_samples_split': 4}
